# Basic cleaning and exploratory data analysis (EDA)
---

## Goals
- Creating environment
- Importing libraries.
- Importing ticker data.
- Importing net income.
- Importing shares outstanding data. 
- Viewing columns, data and data types.
- Cleaning nan or 0 values, removing these valued rows as they are most likely non trading days and would skew the aggregating functions later on.
- Converting column data types for consistency.
---

## Creating environment
```Bash
mamba create -n stock_project python=3.12 yfinance pandas numpy seaborn matplotlib
mamba activate stock_project
```

---
## Importing libraries

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

---
# Importing OHLC & volume data
- Will import
    - HDFC Bank
    - ICICI Bank
    - Bank Nifty
- The time period will be 10 years

In [ ]:
tickers = ['HDFCBANK.NS', 'ICICIBANK.NS', '^NSEBANK']
data = yf.download(tickers, period = "10y", auto_adjust=True)
# auto_adjust=True is for handling stock splits and dividends in the source data.

[*********************100%***********************]  3 of 3 completed


---
## Quick view of the data
- Will see some rows
- Check data types of the columns

In [8]:
print("Data ->")
print(data.head())

Data ->
Price        Adj Close                                  Close               \
Ticker     HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                         
2016-03-21  240.374496   199.372635  15925.914062  261.399994   213.500000   
2016-03-22  242.271088   198.311417  15935.814453  263.462494   212.363632   
2016-03-23  241.236603   198.820801  15887.565430  262.337494   212.909088   
2016-03-28  240.811310   191.392624  15604.718750  261.875000   204.954544   
2016-03-29  242.259598   189.652298  15666.068359  263.450012   203.090912   

Price                           High                                    Low  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2016-03-21  15926.099609  261.837494   214.181824  15945.799805  258.062500   
2016-03-22  15936.000000  264.149994   213.272720  

In [9]:
print(data.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 2471 entries, 2016-03-21 to 2026-03-20
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   (Adj Close, HDFCBANK.NS)   2471 non-null   float64
 1   (Adj Close, ICICIBANK.NS)  2471 non-null   float64
 2   (Adj Close, ^NSEBANK)      2467 non-null   float64
 3   (Close, HDFCBANK.NS)       2471 non-null   float64
 4   (Close, ICICIBANK.NS)      2471 non-null   float64
 5   (Close, ^NSEBANK)          2467 non-null   float64
 6   (High, HDFCBANK.NS)        2471 non-null   float64
 7   (High, ICICIBANK.NS)       2471 non-null   float64
 8   (High, ^NSEBANK)           2467 non-null   float64
 9   (Low, HDFCBANK.NS)         2471 non-null   float64
 10  (Low, ICICIBANK.NS)        2471 non-null   float64
 11  (Low, ^NSEBANK)            2467 non-null   float64
 12  (Open, HDFCBANK.NS)        2471 non-null   float64
 13  (Open, ICICIBANK.NS)       2471 non-null 

---
## Checking for nan values & 0 values

In [10]:
print("How many nan values -> ")
print(data.isna().sum())
print("How many 0 values -> ")
print((data == 0).sum())

How many nan values -> 
Price      Ticker      
Adj Close  HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        4
Close      HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        4
High       HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        4
Low        HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        4
Open       HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        4
Volume     HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        4
dtype: int64
How many 0 values -> 
Price      Ticker      
Adj Close  HDFCBANK.NS       0
           ICICIBANK.NS      0
           ^NSEBANK          0
Close      HDFCBANK.NS       0
           ICICIBANK.NS      0
           ^NSEBANK          0
High       HDFCBANK.NS       0
           ICICIBANK.NS      0
           ^NSEBANK          0
Low        HDFCBANK.NS       0
           ICICIBANK.NS      0
           ^NSEBANK          0

### Insights
- So there are only 4 nan values throughout.
- There are no 0 values in the stock prices which is a good thing, but there are 918 in volume for ^NSEBANK, and 2 each for the banks. This is most likely due to these being no trading days and rather than using nan, 0 volume was represented. There is no 0 or nan in the stock prices of the banks as they are using forward filling with the close of the last trading day for getting the stock prices.

### What to do?
- Will drop the nan values and the 0 values based on NSEBANK as it is the most major instrument and if there is no volume in it then it is most likely to be a non trading day.

---
## Dropping rows with nan and 0 values

### Dropping 0 values rows

In [11]:
data_dropped = data[~(data == 0).any(axis=1)].copy()
print("How many 0 values -> ")
print((data_dropped == 0).sum())


How many 0 values -> 
Price      Ticker      
Adj Close  HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Close      HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
High       HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Low        HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Open       HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Volume     HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
dtype: int64


### Dropping nan valued rows

In [12]:
data_dropped = data_dropped[~data_dropped.isna().any(axis=1)].copy()
print("How many nan values -> ")
print(data_dropped.isna().sum())

How many nan values -> 
Price      Ticker      
Adj Close  HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Close      HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
High       HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Low        HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Open       HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
Volume     HDFCBANK.NS     0
           ICICIBANK.NS    0
           ^NSEBANK        0
dtype: int64


In [13]:
data_dropped.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1548 entries, 2016-03-21 to 2026-03-19
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   (Adj Close, HDFCBANK.NS)   1548 non-null   float64
 1   (Adj Close, ICICIBANK.NS)  1548 non-null   float64
 2   (Adj Close, ^NSEBANK)      1548 non-null   float64
 3   (Close, HDFCBANK.NS)       1548 non-null   float64
 4   (Close, ICICIBANK.NS)      1548 non-null   float64
 5   (Close, ^NSEBANK)          1548 non-null   float64
 6   (High, HDFCBANK.NS)        1548 non-null   float64
 7   (High, ICICIBANK.NS)       1548 non-null   float64
 8   (High, ^NSEBANK)           1548 non-null   float64
 9   (Low, HDFCBANK.NS)         1548 non-null   float64
 10  (Low, ICICIBANK.NS)        1548 non-null   float64
 11  (Low, ^NSEBANK)            1548 non-null   float64
 12  (Open, HDFCBANK.NS)        1548 non-null   float64
 13  (Open, ICICIBANK.NS)       1548 non-null 

---
## Converting the HDFC & ICICI bank volume to float for consistency

In [14]:
data_cleaned = data_dropped.copy()
data_cleaned['Volume'] = data_cleaned['Volume'].astype('float64').copy()
data_cleaned.info()
data_cleaned.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 1548 entries, 2016-03-21 to 2026-03-19
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   (Adj Close, HDFCBANK.NS)   1548 non-null   float64
 1   (Adj Close, ICICIBANK.NS)  1548 non-null   float64
 2   (Adj Close, ^NSEBANK)      1548 non-null   float64
 3   (Close, HDFCBANK.NS)       1548 non-null   float64
 4   (Close, ICICIBANK.NS)      1548 non-null   float64
 5   (Close, ^NSEBANK)          1548 non-null   float64
 6   (High, HDFCBANK.NS)        1548 non-null   float64
 7   (High, ICICIBANK.NS)       1548 non-null   float64
 8   (High, ^NSEBANK)           1548 non-null   float64
 9   (Low, HDFCBANK.NS)         1548 non-null   float64
 10  (Low, ICICIBANK.NS)        1548 non-null   float64
 11  (Low, ^NSEBANK)            1548 non-null   float64
 12  (Open, HDFCBANK.NS)        1548 non-null   float64
 13  (Open, ICICIBANK.NS)       1548 non-null 

Price        Adj Close                                  Close               \
Ticker     HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                         
2016-03-21  240.374496   199.372635  15925.914062  261.399994   213.500000   
2016-03-22  242.271088   198.311417  15935.814453  263.462494   212.363632   
2016-03-23  241.236603   198.820801  15887.565430  262.337494   212.909088   
2016-03-28  240.811310   191.392624  15604.718750  261.875000   204.954544   
2016-03-29  242.259598   189.652298  15666.068359  263.450012   203.090912   

Price                           High                                    Low  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2016-03-21  15926.099609  261.837494   214.181824  15945.799805  258.062500   
2016-03-22  15936.000000  264.149994   213.272720  15967.849609  260.174988   
2016-03-23  15887.750000  263.262512   213.636368  15919.599609  260.137512   
2016-03-28  15604.900391  263.250000   212.727264  15890.700195  260.412506   
2016-03-29  15666.250000  264.875000   206.318176  15774.099609  261.062500   

Price                                        Open                             \
Ticker     ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK   
Date                                                                           
2016-03-21   210.409088  15735.549805  258.250000   210.500000  15735.549805   
2016-03-22   209.636368  15779.150391  261.625000   213.272720  15930.450195   
2016-03-23   209.818176  15799.950195  263.262512   210.409088  15884.250000   
2016-03-28   203.545456  15522.200195  261.250000   212.727264  15867.750000   
2016-03-29   202.181824  15580.799805  261.875000   203.000000  15583.099609   

Price           Volume                        
Ticker     HDFCBANK.NS ICICIBANK.NS ^NSEBANK  
Date                                          
2016-03-21   5318796.0   14818419.0  73600.0  
2016-03-22   4120264.0   12269251.0  58600.0  
2016-03-23   8464300.0   16987529.0  64100.0  
2016-03-28  11270868.0   20214214.0  83500.0  
2016-03-29   6733952.0   20447782.0  69500.0